In [ ]:
import tensorflow as tf

In [ ]:
print(tf.config.list_physical_devices('GPU'))
# testing to see if the gpu and cpu drivers are working with the tenserflow or not

In [ ]:
!nvidia-smi

In [ ]:
import os
import librosa
import tensorflow as tf
from tensorflow.image import resize
import numpy as np

In [ ]:
model = tf.keras.models.load_model("Trained_model.h5")

In [ ]:
model.summary()

In [ ]:
classes = ['blues', 'classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']

## Single Audio Prediction

### Single Audio Preprocessing
before feeding the new file to the model we have to preprocess the file in the same format as our model is trained
it's the same concept of standardization in the linear regression project where we use scaler for the new prediction
before giving it to the model

In [ ]:
def load_and_preprocess_data(file_path,target_shape = (150,150)):
    data = []
    audio_data, sample_rate = librosa.load(file_path, sr = None)

    #define the duration of each chunk and overlap
    chunk_duration = 4
    overlap_duration = 2
  
    #convert duration to sample
    chunk_samples = chunk_duration*sample_rate
    overlap_samples = overlap_duration*sample_rate 

    #calculate  the number of chunks
    num_chunks = int(np.ceil((len(audio_data) - chunk_samples)/(chunk_samples - overlap_samples))) +1
    
    # Iterate over each chunk
    for i in range(num_chunks):
        # Calculate start and end indices of the chunk
        start = i * (chunk_samples - overlap_samples)
        end = start + chunk_samples
        
        # Extract the chunk of audio
        chunk = audio_data[start:end]

        # melspectrogram part, this is the matrix we are getting the 
        mel_spectrogram = librosa.feature.melspectrogram(y = chunk, sr = sample_rate) #calculating spectrogram by this

        #resizing matrix on given target shape as written in the function arguments, you can see the size of melsprectorgram
        # in the visualization part above

        mel_spectrogram = resize(np.expand_dims(mel_spectrogram, axis = -1), target_shape)

        # appending the data to the lists
        data.append(mel_spectrogram)
                    
    #return with typecast to np array                
    return np.array(data)    

### Playing the sound

In [ ]:
from IPython.display import Audio
file_path = "./Test Music/classical-piano-9316.mp3"
# bensound-allthewayup-rock.mp3
# bensound-jazz-lefacteur.mp3
# classical-piano-9316.mp3
# move-forward-hip-hop-165711.mp3
# hindi_classical_1.mp3 (out of the box classical music)
y, sr = librosa.load(file_path, sr = None)
Audio(data= y, rate = sr)

In [ ]:
X_test = load_and_preprocess_data(file_path)

In [ ]:
X_test.shape

In [ ]:
len(y)/sr

### Model Prediction

In [ ]:
def model_prediction(X_test):
    y_pred = model.predict(X_test) #-> hot encoded probability of 10 classes for 55 chunks
    predicted_categories = np.argmax(y_pred, axis = 1) #-> index of each class for 55 chunks(1D array)
    unique_elements, counts = np.unique(predicted_categories, return_counts = True)
    max_element = np.max(counts)
    return unique_elements[counts == max_element][0]
    # will return the index of the predicted class

In [ ]:
result = model_prediction(X_test)

In [ ]:
result # have the index of predicted class

In [ ]:
print(f"Music genre is {classes[result]}")

### Breakdown work of above function

In [ ]:
un, count = model_prediction(X_test)

In [ ]:
un,count

In [ ]:
np.max(count)

In [ ]:
un[count == 49][0]

In [ ]:
np.unique(result, return_counts = True)

In [ ]:
classes # 0-> blues, out of the 55 chunks most are from 0(blues)